# 1. Formulación del problema

In [1]:
# Clase Abstracta
class Problem:
    def __init__(self, initial, goal):
        self.initial = initial
        self.goal = goal

    def actions(self, state):
        raise NotImplementedError

    #Funcion de transicion de estados
    def result(self, state, action):
        raise NotImplementedError

    def is_goal(self, state):
        return self.goal == state

    def action_cost(self, state1, action, state2):
        return 1

    def h(self, state):
        return 0

In [2]:
class GraphProblem(Problem):
    def __init__(self, initial, goal, graph):
        super().__init__(initial, goal)
        self.graph = graph

    def actions(self, state):
        return list(self.graph.get(state, {}).keys())

    def result(self, state, action):
        return action

    def action_cost(self, state1, action, state2):
        return 1

In [3]:
class graph (Problem):
    
    def __init__(self, initial, goal, graph):
        super().__init__(initial, goal)
        self.graph = graph

    def actions(self, state):
        lista = []
        for key in self.graph[state].keys():
            lista.append(key)
        return lista
        
    def result(self, state, action):
        return action

    def action_cost(self, state1, action, state2):
        return self.graph[state1][state2]

In [4]:
romania = {
    'Arad': {'Zerind': 75, 'Sibiu': 140, 'Timisoara': 118},
    'Zerind': {'Arad': 75, 'Oradea': 71},
    'Oradea': {'Zerind': 71, 'Sibiu': 151},
    'Sibiu': {'Arad': 140, 'Oradea': 151, 'Fagaras': 99, 'Rimnicu Vilcea': 80},
    'Timisoara': {'Arad': 118, 'Lugoj': 111},
    'Lugoj': {'Timisoara': 111, 'Mehadia': 70},
    'Mehadia': {'Lugoj': 70, 'Dobreta': 75},
    'Dobreta': {'Mehadia': 75, 'Craiova': 120},
    'Craiova': {'Dobreta': 120, 'Rimnicu Vilcea': 146, 'Pitesti': 138},
    'Rimnicu Vilcea': {'Sibiu': 80, 'Craiova': 146, 'Pitesti': 97},
    'Fagaras': {'Sibiu': 99, 'Bucarest': 211},
    'Pitesti': {'Rimnicu Vilcea': 97, 'Craiova': 138, 'Bucarest': 101},
    'Bucarest': {'Fagaras': 211, 'Pitesti': 101, 'Giurgiu': 90, 'Urziceni': 85},
    'Giurgiu': {'Bucarest': 90},
    'Urziceni': {'Bucarest': 85, 'Hirsova': 98, 'Vaslui': 142},
    'Hirsova': {'Urziceni': 98, 'Eforie': 86},
    'Eforie': {'Hirsova': 86},
    'Vaslui': {'Urziceni': 142, 'Iasi': 92},
    'Iasi': {'Vaslui': 92, 'Neamt': 87},
    'Neamt': {'Iasi': 87},
}

In [5]:
class Node:
    def __init__(self, state, parent=None, action=None, path_cost=0):
        self.state = state
        self.parent = parent
        self.action = action
        self.path_cost = path_cost

    def path(self):
        lista_path = []
        node = self
        while node:
            lista_path.append(node.state)
            node = node.parent
        return lista_path[::-1]

    def expand(self, problem):
        lista = []
        for action in problem.actions(self.state):
            lista.append(self.child_node(problem, action))
        return lista

    def child_node(self, problem, action):
        next_state = problem.result(self.state, action)
        step_cost = problem.action_cost(self.state, action, next_state)
        return Node(next_state, self, action, self.path_cost + step_cost)

In [6]:
def depth_first_graph_search(problem):
    node = Node(problem.initial)
    if problem.is_goal(node.state):
        return node
    frontier = [node]
    explored = set()
    while frontier:
        node = frontier.pop()
        explored.add(node.state)
        for child in node.expand(problem):
            if child.state not in explored and child not in frontier:
                if problem.is_goal(child.state):
                    return child
                frontier.append(child)
    return None

In [7]:
problem = GraphProblem('Arad', 'Bucarest', romania)
result_node = depth_first_graph_search(problem)
node = result_node
path = node.path()
print("DFS Path from Arad to Bucarest:", path)

DFS Path from Arad to Bucarest: ['Arad', 'Timisoara', 'Lugoj', 'Mehadia', 'Dobreta', 'Craiova', 'Pitesti', 'Bucarest']


In [8]:
from collections import deque

def breadth_first_graph_search(problem):
    node = Node(problem.initial)

    if problem.is_goal(node.state):
        return node

    frontier = deque([node])
    frontier_states = {node.state}
    explored = set()

    while frontier:
        node = frontier.popleft()
        frontier_states.remove(node.state)
        explored.add(node.state)

        for child in node.expand(problem):
            if (
                child.state not in explored
                and child.state not in frontier_states
            ):
                if problem.is_goal(child.state):
                    return child

                frontier.append(child)
                frontier_states.add(child.state)

    return None

In [9]:
problem1 = GraphProblem('Arad', 'Bucarest', romania)

result_node = breadth_first_graph_search(problem1)

if result_node is not None:
    print("Nodo encontrado:", result_node.state)
    print("Ruta BFS:", result_node.path())
    print("Número de pasos:", len(result_node.path()) - 1)
    print("Costo registrado:", result_node.path_cost)
else:
    print("No se encontró una ruta.")

Nodo encontrado: Bucarest
Ruta BFS: ['Arad', 'Sibiu', 'Fagaras', 'Bucarest']
Número de pasos: 3
Costo registrado: 3
